# Notebook 24 — When to use embeddings alongside pathway scores

PSF's v0.6 Phase 2 `pathway_subtyping.embed` layer provides stable
wrappers around cell-foundation models. The first supported backend is
scGPT (Cui et al. 2024); the same interface covers UCE (F2) and
Geneformer (F5).

**Pathway scores are still the primary output.** Embeddings are a
complement: useful for rare-cell detection, trajectory inference, and
cross-platform harmonization (F2) where pathway scores alone miss
relevant structure.

Research use only. Not for clinical decision-making.

In [ ]:
import numpy as np
import pandas as pd
from pathway_subtyping.embed import (
    scGPTEmbedder, FallbackSCGPTEmbedder,
    EmbeddingCache, cache_key_for,
)

rng = np.random.default_rng(0)
n_cells, n_genes = 200, 50
expression = pd.DataFrame(
    rng.standard_normal((n_cells, n_genes)),
    columns=[f'GENE_{i}' for i in range(n_genes)],
    index=[f'cell_{i}' for i in range(n_cells)],
)

## 1. Embed cells with scGPT (or the fallback)

In [ ]:
embedder = scGPTEmbedder(FallbackSCGPTEmbedder(embedding_dim=32))
result = embedder.embed(expression)
print(f'backend: {result.backend}')
print(f'embeddings shape: {result.embeddings.shape}')
result.as_dataframe().head()

## 2. Cache embeddings across reruns

The cache key covers the backend identifier and the bytes of the input
expression matrix. Changing either produces a fresh key; changing
nothing hits the cache.

In [ ]:
import tempfile, pathlib
with tempfile.TemporaryDirectory() as tmp:
    cache = EmbeddingCache(pathlib.Path(tmp))
    key = cache_key_for(backend=result.backend, expression=expression)
    cache.put(key, result)
    print('cache hit:', cache.has(key))
    reloaded = cache.get(key)
    print('reloaded shape:', reloaded.embeddings.shape)

## 3. Drop embeddings into the F2 harmonize layer

`scGPTEmbedder.embed` returns an `EmbeddingResult` whose
`embeddings` ndarray is a drop-in substitute for UCE's embedding in
`CrossPlatformAligner`. Same interface, different biology.

In [ ]:
from pathway_subtyping.harmonize import CrossPlatformAligner

half = n_cells // 2
platforms = ['10x'] * half + ['smartseq2'] * (n_cells - half)
pathway_scores = pd.DataFrame(
    rng.standard_normal((n_cells, 5)),
    index=expression.index,
    columns=[f'PATH_{i}' for i in range(5)],
)
aligned = CrossPlatformAligner().fit_transform(
    pathway_scores, platforms, result.embeddings,
)
aligned.aligned_scores.head()

## Further reading

- Cui H et al. (2024). *scGPT: toward building a foundation model for
  single-cell multi-omics using generative AI.* Nature Methods.
- PSF v0.6 roadmap — Phase 2 F6:
  [docs/roadmap-v06-codeberg.md](../../docs/roadmap-v06-codeberg.md)
- Cross-platform guide: [docs/guides/cross-platform.md](../../docs/guides/cross-platform.md)